# Module 01 — Classic CNN Architectures

We trace the architectural lineage that shaped modern computer vision:
**LeNet → AlexNet → VGG → ResNet**.

**Topics**
1. LeNet-5 on MNIST
2. AlexNet and deep networks
3. VGG philosophy — stacking 3×3 convolutions
4. Vanishing gradients demonstration
5. ResNet: residual connections
6. Transfer learning with a pretrained ResNet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from models import LeNet5, AlexNet, VGG, BasicBlock, Bottleneck, ResNet, resnet18, resnet50

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. LeNet-5 on MNIST

LeNet-5 (1998) is the ancestor of all modern CNNs.  Its key innovation was
*weight sharing* — the same filter is applied at every spatial location,
drastically reducing the number of parameters compared to a fully-connected network.

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.Resize(32),  # LeNet expects 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = datasets.MNIST('data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST('data', train=False, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)
test_loader   = DataLoader(test_dataset,  batch_size=1000, shuffle=False, num_workers=0)

# Inspect a batch
imgs, labels = next(iter(train_loader))
print('Batch shape:', imgs.shape)  # (256, 1, 32, 32)
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for ax, img, lbl in zip(axes, imgs, labels):
    ax.imshow(img.squeeze(), cmap='gray'); ax.set_title(str(lbl.item())); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
model = LeNet5().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'LeNet-5 parameters: {total_params:,}')  # ~44k
print(model)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = model(imgs).argmax(dim=1)
        correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)

train_losses = []
for epoch in range(3):
    loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    acc  = evaluate(model, test_loader, DEVICE)
    train_losses.append(loss)
    print(f'Epoch {epoch+1:02d} | loss={loss:.4f} | test_acc={acc:.4f}')

## 4. Vanishing Gradients

In deep networks without residual connections, gradients shrink exponentially
as they propagate back through layers (especially with sigmoid activations).
Let us measure the gradient norm at each layer of a deep sequential network.

In [ ]:
# Build a deep plain network (no residual connections)
class PlainNet(nn.Module):
    def __init__(self, depth=20, use_sigmoid=True):
        super().__init__()
        act = nn.Sigmoid() if use_sigmoid else nn.ReLU()
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(64, 64), act]
        self.net = nn.Sequential(nn.Linear(784, 64), act, *layers, nn.Linear(64, 10))
    def forward(self, x):
        return self.net(x.flatten(1))

plain = PlainNet(depth=20, use_sigmoid=True).to(DEVICE)
imgs, labels = next(iter(train_loader))
imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
loss = nn.CrossEntropyLoss()(plain(imgs), labels)
loss.backward()

grad_norms = []
for name, p in plain.named_parameters():
    if p.grad is not None and 'weight' in name:
        grad_norms.append(p.grad.norm().item())

plt.figure(figsize=(10, 3))
plt.semilogy(grad_norms, 'o-')
plt.xlabel('Layer index (output → input)')
plt.ylabel('Gradient L2 norm (log scale)')
plt.title('Vanishing gradients in a 20-layer plain network (Sigmoid)')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. ResNet — Residual Connections

The key identity: `output = F(x) + x`.  The gradient of the loss with respect
to `x` is `∂L/∂x = ∂L/∂output × (∂F/∂x + I)`.  The identity term `I` ensures
gradients always have a direct path backwards, regardless of `F`.

In [ ]:
# Parameter counts
for name, fn in [('ResNet-18', lambda: resnet18(1000)),
                  ('ResNet-50', lambda: resnet50(1000))]:
    m = fn()
    n = sum(p.numel() for p in m.parameters())
    print(f'{name}: {n/1e6:.1f}M parameters')

# Forward pass shape check
net = resnet18(10)
x = torch.randn(2, 3, 224, 224)
y = net(x)
print('ResNet-18 output shape:', y.shape)  # (2, 10)

## 6. Transfer Learning

We load a ResNet-18 pretrained on ImageNet, freeze the backbone, replace the
head, and fine-tune on a tiny 10-way CIFAR-10 subset.

In [ ]:
# Load pretrained ResNet-18
backbone = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all layers
for param in backbone.parameters():
    param.requires_grad = False

# Replace classification head
backbone.fc = nn.Linear(backbone.fc.in_features, 10)
backbone = backbone.to(DEVICE)

# Only the new head will be trained
trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total     = sum(p.numel() for p in backbone.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

# CIFAR-10 data
cifar_tfm = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
cifar_train = datasets.CIFAR10('data', train=True, download=True, transform=cifar_tfm)
# Use a tiny subset for speed
subset = torch.utils.data.Subset(cifar_train, range(500))
loader = DataLoader(subset, batch_size=32, shuffle=True)

opt = optim.Adam(backbone.fc.parameters(), lr=1e-3)
for epoch in range(2):
    loss_val = train_one_epoch(backbone, loader, opt, nn.CrossEntropyLoss(), DEVICE)
    print(f'Epoch {epoch+1} | loss={loss_val:.4f}')

print('Transfer learning demo complete.')

## Exercise — Build Your Own Residual Block

Implement a `PreActivationBlock` (He et al., 2016, "Identity Mappings in Deep
Residual Networks") where the order is:

```
BN → ReLU → Conv → BN → ReLU → Conv
```

with the same shortcut connection as `BasicBlock`.

In [ ]:
### EXERCISE — Pre-activation residual block

class PreActivationBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        # TODO: define layers
        raise NotImplementedError

    def forward(self, x):
        # TODO: implement forward pass
        raise NotImplementedError

# Test: output shape should match input when stride=1
# block = PreActivationBlock(64, 64)
# x = torch.randn(2, 64, 8, 8)
# print(block(x).shape)  # should be torch.Size([2, 64, 8, 8])